### Firewall API - Nested & Parameterized Resources

[![Python 3.11+](https://img.shields.io/badge/Python-3.11%2B-blue.svg)](https://www.python.org/downloads/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](../LICENSE)
[![Development Status: Beta 0.8.0](https://img.shields.io/badge/Status-Beta%200.8.0-orange.svg)](https://pypi.org/project/sfos-api/)
[![Code Style: Ruff](https://img.shields.io/endpoint?url=https://raw.githubusercontent.com/astral-sh/ruff/main/assets/badge/v2.json)](https://github.com/astral-sh/ruff)

**Purpose:** Read, update, and execute item-level actions on nested and path-parameterized resources using domain StrEnum templates.

**Topics:**
- In-path template substitution using domain StrEnum objects (`Logging`, `Authentication`, `ActiveThreatResponse`, `DenialOfService`)
- Reading and updating syslog server log component settings (`Logging.LOG_DESTINATIONS_SYSLOG_SERVERS_BY_ID_OR_NAME_LOGS`)
- Fetching Active Directory groups via item-level action (`Authentication.SERVERS_ACTIVE_DIRECTORY_BY_ID_OR_NAME_FETCH_GROUPS`)
- Synchronizing third-party threat feeds (`ActiveThreatResponse.THIRDPARTY_THREAT_FEEDS_BY_ID_OR_NAME_SYNCHRONIZE`)
- Reading flood attack statistics by custom path key (`DenialOfService.ATTACKS_BY_FLOOD_ATTACK_TYPE`)
- Error handling

**Requirements:**
- Credentials configured (`.credentials/config.ini`)
- SFOS firewall accessible
- API key with appropriate permissions

---

#### Setup: Import Libraries and Configure Client

In [ ]:
from utils_api import find_credentials_file, print_data

from sfos_api import (
    ActiveThreatResponse,
    Authentication,
    DenialOfService,
    Logging,
    NotFoundError,
    SFOSApiError,
    SFOSClient,
)

# Initialize the SFOS API Client
client = SFOSClient(config_path=find_credentials_file())
print("✅ SFOS API client initialized")

#### 2. Read Syslog Server Logs Configuration (Sub-Resource GET)

Using `Logging.LOG_DESTINATIONS_SYSLOG_SERVERS_BY_ID_OR_NAME_LOGS` with `id_or_name` seamlessly substitutes the server ID into the sub-resource path.

---

In [ ]:
endpoint = Logging.LOG_DESTINATIONS_SYSLOG_SERVERS_BY_ID_OR_NAME_LOGS
server_name = "firewall"

try:
    # Read sub-resource using Endpoint constant
    response = client.get(endpoint, id_or_name=server_name)
    print(f"✅ Fetched log component settings for '{server_name}':")
    print_data(response)
except NotFoundError:
    print(f"❌ Not Found (HTTP 404): Syslog server '{server_name}' does not exist on the firewall.")
    print("   Guidance: Ensure a syslog server is created under /logging/log-destinations/syslog-servers first.")
except SFOSApiError as e:
    print(f"❌ SFOS API Error [HTTP {e.status_code}]: {e}")
    if e.response_body:
        print_data(e.response_body)
except Exception as e:
    print(f"❌ Unexpected Error ({type(e).__name__}): {e}")

#### 3. Update Syslog Server Logs Configuration (Sub-Resource PATCH)

Update log components using the typed Endpoint constant.

---

In [ ]:
payload = {
    "logComponents": [
        {"name": "firewallRules", "enabled": True},
        {"name": "anomaly", "enabled": True},
    ],
}

try:
    # Update sub-resource using Endpoint constant
    response = client.patch(endpoint, id_or_name=server_name, data=payload)
    print(f"✅ Updated log components for '{server_name}':")
    print_data(response)
except NotFoundError:
    print(f"❌ Not Found (HTTP 404): Cannot update logs; syslog server '{server_name}' does not exist.")
    print("   Guidance: Ensure the target syslog server is created before updating its log components.")
except SFOSApiError as e:
    print(f"❌ SFOS API Error [HTTP {e.status_code}]: {e}")
    if e.response_body:
        print_data(e.response_body)
except Exception as e:
    print(f"❌ Unexpected Error ({type(e).__name__}): {e}")

#### 4. Fetch Active Directory Groups (Item-Level Action POST)

Using `Authentication.SERVERS_ACTIVE_DIRECTORY_BY_ID_OR_NAME_FETCH_GROUPS` with `id_or_name`.

---

In [ ]:
ad_endpoint = Authentication.SERVERS_ACTIVE_DIRECTORY_BY_ID_OR_NAME_FETCH_GROUPS
ad_server_name = "AD"

action_payload = {
    "baseDN": "dc=sfos,dc=local",
}

try:
    # Execute action via Endpoint constant with id_or_name
    result = client.post(ad_endpoint, id_or_name=ad_server_name, data=action_payload)
    print(f"✅ Fetched AD groups for '{ad_server_name}':")
    print_data(result)
except NotFoundError:
    print(f"❌ Not Found (HTTP 404): Active Directory server '{ad_server_name}' does not exist on the firewall.")
    print("   Guidance: Configure the Active Directory server under /authentication/servers/active-directory first.")
except SFOSApiError as e:
    print(f"❌ SFOS API Error [HTTP {e.status_code}]: {e}")
    if e.response_body:
        print_data(e.response_body)
except Exception as e:
    print(f"❌ Unexpected Error ({type(e).__name__}): {e}")

#### 5. Synchronize Third-Party Threat Feed (Item Action POST)
Using `ActiveThreatResponse.THIRDPARTY_THREAT_FEEDS_BY_ID_OR_NAME_SYNCHRONIZE`.

---

In [ ]:
sync_endpoint = ActiveThreatResponse.THIRDPARTY_THREAT_FEEDS_BY_ID_OR_NAME_SYNCHRONIZE
feed_name = "Feodor"

try:
    # Trigger threat feed sync using Endpoint constant
    result = client.post(sync_endpoint, id_or_name=feed_name)
    print(f"✅ Threat feed '{feed_name}' synchronized:")
    print_data(result)
except NotFoundError:
    print(f"❌ Not Found (HTTP 404): Third-party threat feed '{feed_name}' does not exist.")
    print(
        "   Guidance: Add a threat feed under /active-threat-response/thirdparty-threat-feeds before triggering sync."
    )
except SFOSApiError as e:
    print(f"❌ SFOS API Error [HTTP {e.status_code}]: {e}")
    if e.response_body:
        print_data(e.response_body)
except Exception as e:
    print(f"❌ Unexpected Error ({type(e).__name__}): {e}")

#### 6. Read Denial-of-Service Attack Statistics (Custom Keyed Item GET)

Using `DenialOfService.ATTACKS_BY_FLOOD_ATTACK_TYPE`.

---

In [ ]:
dos_endpoint = DenialOfService.ATTACKS_BY_FLOOD_ATTACK_TYPE
attack_type = "SYN"  # Value must be 'SYN', 'UDP', 'ICMP', or 'IP'.

try:
    # Read attack statistics for specific flood type
    result = client.get(dos_endpoint, id_or_name=attack_type)
    print(f"✅ DoS attack statistics for '{attack_type}':")
    print_data(result)
except NotFoundError:
    print(f"❌ Not Found (HTTP 404): Attack type '{attack_type}' was not recognized by the firewall.")
    print("   Guidance: Valid flood attack types are 'SYN', 'UDP', 'ICMP', or 'IP'.")
except SFOSApiError as e:
    print(f"❌ SFOS API Error [HTTP {e.status_code}]: {e}")
    if e.response_body:
        print_data(e.response_body)
except Exception as e:
    print(f"❌ Unexpected Error ({type(e).__name__}): {e}")

In [ ]:
# Close the client session when done
client.close()